In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from regions import Regions
import regions
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.modeling import models, fitting

from dust_extinction.averages import CT06_MWGC

from smart_plotters.jwst_plots import JWSTCatalog, make_cat_use, make_brick_cat
from smart_plotters.cutout_plot import get_cutout_405, get_cutout_jwst_ice
import smart_plotters.co_ice as co_map
from smart_plotters import cmd_plot 

from mpl_plot_templates import adaptive_param_plot

#get_dmag_tbl

ImportError: cannot import name 'get_dmag_tbl' from 'smart_plotters.jwst_plots' (/blue/adamginsburg/savannahgramze/cloudc/smart-plotters/smart_plotters/jwst_plots.py)

In [2]:
def get_rc_sel_mask(cat):
    x = np.linspace(0, 2.5, 10)
    
    x0 = 0.52
    mask_x0_left = cat.color('f182m', 'f212n') > x0

    y0 = 14.8
    mask_above_y0 = cat.band('f182m') > y0
    mask_rc = mask_above_y0 & mask_x0_left

    pt1 = (0.5, 14.3)
    pt2 = (2.0, 20.)
    y1 = (pt2[1] - pt1[1]) / (pt2[0] - pt1[0]) * (x - pt1[0]) + pt1[1]
    mask_below_y1 = cat.band('f182m') > ( (pt2[1] - pt1[1]) / (pt2[0] - pt1[0]) * (cat.color('f182m', 'f212n') - pt1[0]) + pt1[1] )
    mask_rc = mask_below_y1 & mask_rc

    pt1 = (0.5, 15.5)
    pt2 = (2.0, 20.9)
    y2 = (pt2[1] - pt1[1]) / (pt2[0] - pt1[0]) * (x - pt1[0]) + pt1[1]
    mask_above_y2 = cat.band('f182m') < ( (pt2[1] - pt1[1]) / (pt2[0] - pt1[0]) * (cat.color('f182m', 'f212n') - pt1[0]) + pt1[1] )
    mask_rc = mask_rc & mask_above_y2

    return mask_rc

# Model the CCDs for different ice compositions

In [4]:
tbl = co_map.get_dmag_tbl()

In [10]:
tbl_sel = tbl.loc['H2O:CO:CO2 (10:1:1)']

In [11]:
tbl_sel

molecule,mol_id,molwt,database,author,composition,temperature,density,column,F115W,F150W,F162M,F182M,F187N,F200W,F210M,F212N,F250M,F277W,F300M,F323N,F322W2,F335M,F356W,F360M,F405N,F410M,F430M,F444W,F460M,F466N,F470N,F480M,F560W,F770W,F1000W,F1065C,F1130W,F1140C,F1280W,F1500W,F1550C,F1800W,F2100W,F2300C,F2550W
,,u,,,,K,g / cm3,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
str25,int64,float64,str5,str82,str66,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
h2o:co:co2,19,21.0,mymix,"Mastrapa, Gerakines2023, Gerakines",H2O:CO:CO2 (10:1:1),25.0,1.0,1e+17,1.5573683835867769e-06,4.6207030793610215e-05,3.665931670404632e-05,5.1505520584527176e-05,2.2062928126231895e-05,0.00010321178109862217,0.00014637951094265134,9.097041131411743e-05,9.238121513988062e-05,0.015608115704129233,0.04087495064697677,0.013977065904265729,0.00973211938195817,0.0057804230838947035,0.00435494975718953,0.0002246753262813428,0.0008754215637925,0.0071730886367440405,0.01726402082240064,0.005090694155819975,0.00233382861820175,0.0042613484832187964,0.0016992746370103617,0.001400171192253552,0.0021924660948435815,0.0018876606346687197,0.0019903578370090713,0.003240246924271162,0.00649108585545477,0.006520157889605471,0.00944721233951995,0.009133444271622437,0.00934396250930547,0.0035148125717761047,0.0018548801397653136,0.0007895598007738869,1.090948114068624e-06
h2o:co:co2,19,21.0,mymix,"Mastrapa, Gerakines2023, Gerakines",H2O:CO:CO2 (10:1:1),25.0,1.0,1.2067926406393264e+17,1.8794204930117075e-06,5.576217200342626e-05,4.4240151591878885e-05,6.21558280489154e-05,2.6625373049427026e-05,0.0001245541632144409,0.0001766488819594514,0.00010978239995829142,0.00011148495808832593,0.018786473222977662,0.049306588274312446,0.016866348370424333,0.011711771625975587,0.006966700708145979,0.005244411704023122,0.00027113459859329225,0.0010564515864182056,0.008638601349794328,0.020807474297281914,0.006131529076894182,0.002815049603853481,0.005137861070101835,0.002049773578914227,0.0016887859069552036,0.0026455369478881607,0.0022779949905569197,0.0024018288327543047,0.003910224850898203,0.00783329717630643,0.00786840954219059,0.011400759182295772,0.011020594308343235,0.011273605890956162,0.004241602432612623,0.002238341357003293,0.0009527339723245376,1.3163285572659333e-06
h2o:co:co2,19,21.0,mymix,"Mastrapa, Gerakines2023, Gerakines",H2O:CO:CO2 (10:1:1),25.0,1.0,1.4563484775012384e+17,2.268070517175147e-06,6.72931855341119e-05,5.338862816905987e-05,7.500824175821208e-05,3.213129515700075e-05,0.00015030951161065786,0.00021317735527937032,0.00013248455894654398,0.00013453920879236136,0.02259979746947849,0.059472199947229853,0.020352624110774542,0.01408598872141198,0.008394183285112433,0.006312799374228462,0.00032720042611522615,0.0012749169595203114,0.010399114176941282,0.025071571482786936,0.007382241826281444,0.003395151461637269,0.006193499574365546,0.0024723470896077515,0.002036663446858711,0.003192155827738574,0.0027490385053869915,0.0028983341125972117,0.004718712189593077,0.009453023437998809,0.00949543821743859,0.013758254483430932,0.013297273672721488,0.013601092747604682,0.005118665411236734,0.0027010473281166014,0.0011496052846382554,1.5882158823643522e-06
h2o:co:co2,19,21.0,mymix,"Mastrapa, Gerakines2023, Gerakines",H2O:CO:CO2 (10:1:1),25.0,1.0,1.7575106248547965e+17,2.737090364135497e-06,8.120863961025293e-05,6.44289144506871e-05,9.051800466508553e-05,3.8775797285950375e-05,0.00018139017562113224,0.0002572590933933583,0.00015988134209798943,0.00016236090058363573,0.027169371478168003,0.07172593352643197,0.024559123815391715,0.01692974817417081,0.010110898774032151,0.007594873727621021,0.00039485897286084537,0.001538558889343733,0.012512013853662296,0.030199831067474037,0.00888383578166696,0.00409429786146375